![Banner](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/competitions/datacrunch-2/assets/banner.webp)

# DataCrunch 2 — Stacking Ensemble (sized for repeated `train()` calls)

**Why this version looks different from the last one:** Submission #2 was
terminated after 9h10min without finishing even the *first* of 4 base
models, in the *first* of **9 required `train()` calls** (the platform
retrains at every walk-forward step — `Train Frequency: 1`). DataCrunch's
weekly compute quota is ~10 hours total, shared across every run that
week, so a single `train()` call needs to take minutes, not hours.

**What changed to fit that budget:**
1. **Feature selection is now a single, fully vectorized Pearson-correlation
   filter** (`select_features_by_correlation`) instead of a `RandomForest`
   refit inside every CV fold. It runs a single matrix multiply over the
   whole training set — seconds, not hours, even at a few million rows —
   at the cost of missing nonlinear/interaction effects a model-based
   selector would catch. The base models below see the raw filtered
   features and can still learn interactions themselves.
2. **No hyperparameter search inside `train()`.** Searching per call was
   the other main cost driver, multiplied by 9. Instead, `XGB_FIXED_PARAMS`
   / `LGBM_FIXED_PARAMS` hold fixed, capped hyperparameters (bounded
   depth, moderate `n_estimators`) chosen to be fast and reasonable rather
   than exhaustively tuned. There's a separate, clearly-marked **offline
   tuning cell** near the bottom — run it manually against a subsample if
   you want to explore better values, then copy them into the fixed
   dicts. It isn't referenced by `train()`/`infer()`, so it's stripped
   from the actual submission automatically.
3. **Down to 2 base models (XGBoost, LightGBM) + Ridge meta-learner**,
   not 4. `ExtraTrees` and `LinearSVR` are dropped for now — not because
   they're bad ideas, but because every additional base model multiplies
   cost across all 9 `train()` calls, and we don't yet know the real
   per-call runtime at full scale. Once you've confirmed headroom under
   quota with this leaner version, they're easy to add back.
4. Base models now use `n_jobs=-1` directly (no more nested search
   wrapping them, so no oversubscription risk from doing so).
5. Still: `id`/`moon` dropped, every CV split (for the stack's internal
   out-of-fold predictions) grouped by `moon` via precomputed splits,
   scored on Spearman, `# @crunch/keep:on/off` markers around every
   global the runner needs, `crunch_tools.test()` before submitting.

**Please verify the ~10h/week figure and the exact walk-forward step count
on your competition's own resource page** — I'm going from DataCrunch's
general docs, and exact numbers can vary or change.


In [ ]:
# Install the Crunch CLI + LightGBM
%pip install crunch-cli lightgbm --upgrade --quiet --progress-bar off

# Setup your local environment
!crunch setup-notebook datacrunch-2 0JiCmmP21Ca88X8TApRuDHMH --size small

## Imports

In [ ]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from lightgbm import LGBMRegressor  # >= 4.0
from scipy.stats import randint, uniform, spearmanr
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import make_scorer
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

In [ ]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

## Config & helpers

In [ ]:
# @crunch/keep:on
RANDOM_STATE = 0
ID_COLUMNS = ["id", "moon"]
FEATURE_SELECTION_TOP_K = 300
# @crunch/keep:off


def get_feature_columns(df: pd.DataFrame):
    """All Feature_* columns — explicitly excludes id/moon/target so they
    never get fed into the model as if they were predictive features."""
    return [c for c in df.columns if c not in ID_COLUMNS and c != "target"]


def get_model_path(model_directory_path: str) -> str:
    return os.path.join(model_directory_path, "model.joblib")


def spearman(y_true, y_pred) -> float:
    corr, _ = spearmanr(y_true, y_pred)
    return 0.0 if np.isnan(corr) else corr


# @crunch/keep:on
spearman_scorer = make_scorer(spearman, greater_is_better=True)
# @crunch/keep:off


def select_features_by_correlation(X: pd.DataFrame, y: np.ndarray, top_k: int = FEATURE_SELECTION_TOP_K):
    """Fast, fully vectorized Pearson-correlation feature filter — one
    matrix multiply over the whole training set. Replaces a RandomForest
    -based selector that couldn't finish inside the platform's runtime
    quota when refit per CV fold. Trade-off: linear-only, so nonlinear /
    interaction effects are left for the base models to find themselves."""
    X_vals = X.to_numpy(dtype=np.float32)
    y_vals = y.astype(np.float32)
    X_centered = X_vals - X_vals.mean(axis=0)
    y_centered = y_vals - y_vals.mean()
    numerator = X_centered.T @ y_centered
    denom = np.sqrt((X_centered ** 2).sum(axis=0) * (y_centered ** 2).sum()) + 1e-12
    corr = numerator / denom
    top_idx = np.argsort(-np.abs(corr))[:top_k]
    return [X.columns[i] for i in top_idx]

## Base models (fixed hyperparameters — no per-call search)

Capped depth/leaves and a moderate `n_estimators` so a single fit is fast
at full data scale. These are reasonable starting points, not the result
of exhaustive tuning — see the offline tuning cell near the bottom if you
want to improve on them without paying that cost on every `train()` call.

In [ ]:
# @crunch/keep:on
XGB_FIXED_PARAMS = dict(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.7,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

LGBM_FIXED_PARAMS = dict(
    n_estimators=400,
    max_depth=6,
    num_leaves=63,
    learning_rate=0.03,
    subsample=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
# @crunch/keep:off


def build_stack(X, y, groups, n_splits=3) -> StackingRegressor:
    """2-model stack + Ridge meta-learner. The internal out-of-fold
    predictions use precomputed moon-grouped splits, since
    StackingRegressor.fit() has no `groups` argument to forward to a
    GroupKFold splitter directly."""
    stack_cv = list(GroupKFold(n_splits=n_splits).split(X, y, groups=groups))
    return StackingRegressor(
        estimators=[
            ("xgb", xgb.XGBRegressor(**XGB_FIXED_PARAMS)),
            ("lgbm", LGBMRegressor(**LGBM_FIXED_PARAMS)),
        ],
        final_estimator=Ridge(random_state=RANDOM_STATE),
        cv=stack_cv,
        n_jobs=1,  # base learners already use n_jobs=-1 internally; avoid oversubscription
    )

## Train & infer

These two functions are the actual submission entry points, called
`train()` once per walk-forward step and `infer()` once per moon of live
data. Everything above is just setup they rely on.

In [ ]:
def train(
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    model_directory_path: str,
) -> None:
    """Select features, fit the 2-model stack, and persist it for infer()."""

    feature_columns = get_feature_columns(X_train)
    X_full = X_train[feature_columns]
    y = y_train["target"].to_numpy()
    groups = X_train["moon"].to_numpy()  # group folds by week -> no time leakage

    selected_columns = select_features_by_correlation(X_full, y)
    X = X_full[selected_columns]

    stack = build_stack(X, y, groups)
    stack.fit(X, y)

    os.makedirs(model_directory_path, exist_ok=True)
    joblib.dump(
        {"model": stack, "feature_columns": selected_columns},
        get_model_path(model_directory_path),
    )


def infer(
    X_test: pd.DataFrame,
    model_directory_path: str,
) -> pd.DataFrame:
    """Load the persisted stack and score the current moon of data."""

    saved = joblib.load(get_model_path(model_directory_path))
    model, feature_columns = saved["model"], saved["feature_columns"]

    predictions = X_test[["id", "moon"]].copy()
    predictions["prediction"] = model.predict(X_test[feature_columns])
    return predictions

## Load the data

In [ ]:
X_train, y_train, X_test = crunch_tools.load_data()

## Held-out validation (last 50 moons) — and a timing check

This is your best local read on real runtime, though the local sample is
much smaller than the full dataset (13 moons for `X_test`/`y_test`, per
DataCrunch's docs, vs. hundreds in the real run) — treat the `%%time`
output here as a floor, not the true full-scale number.

In [ ]:
%%time
feature_columns = get_feature_columns(X_train)
moons = np.sort(X_train["moon"].unique())
holdout_moons = moons[-50:]

is_holdout = X_train["moon"].isin(holdout_moons)
X_fit, y_fit = X_train.loc[~is_holdout], y_train.loc[~is_holdout]
X_val, y_val = X_train.loc[is_holdout], y_train.loc[is_holdout]

X_fit_full = X_fit[feature_columns]
y_fit_target = y_fit["target"].to_numpy()
groups_fit = X_fit["moon"].to_numpy()

selected_columns = select_features_by_correlation(X_fit_full, y_fit_target)
X_fit_sel = X_fit_full[selected_columns]

stack = build_stack(X_fit_sel, y_fit_target, groups_fit)
stack.fit(X_fit_sel, y_fit_target)

y_val_target = y_val["target"].to_numpy()
val_pred = stack.predict(X_val[selected_columns])
print(f"Holdout Spearman over last {len(holdout_moons)} moons: {spearman(y_val_target, val_pred):.4f}")

for name, base_model in stack.named_estimators_.items():
    base_pred = base_model.predict(X_val[selected_columns])
    print(f"  {name} alone: {spearman(y_val_target, base_pred):.4f}")

## Optional: offline hyperparameter exploration

**Not used by `train()`/`infer()` — run this manually if you want to
explore better hyperparameters, then copy whatever you find into
`XGB_FIXED_PARAMS`/`LGBM_FIXED_PARAMS` above.** Keeping search out of the
submitted `train()` is what keeps each of the 9 walk-forward calls fast;
this cell exists so you don't lose the ability to tune, just the cost of
doing it on every call. Consider running it against a row subsample if
even one search feels slow — it doesn't need the full dataset to point you
toward reasonable hyperparameters.

In [ ]:
%%time
_search_params = {
    "max_depth": randint(3, 9),
    "n_estimators": randint(150, 500),
    "learning_rate": uniform(0.01, 0.15),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
}

_cv = GroupKFold(n_splits=3)
_search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=_search_params,
    n_iter=10,
    cv=_cv,
    scoring=spearman_scorer,
    n_jobs=1,  # xgb already uses n_jobs=-1 internally
    random_state=RANDOM_STATE,
    verbose=1,
)
_search.fit(X_fit_sel, y_fit_target, groups=groups_fit)
print("Best XGBoost params found:", _search.best_params_)
print(f"Best CV Spearman: {_search.best_score_:.4f}")

## Local test

`crunch_tools.test()` runs your `train()` / `infer()` exactly the way the
platform will, and checks the output format before you submit. Always run
this before pushing.

In [ ]:
crunch_tools.test()